In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', context='paper', font_scale=1.2)

ROOT     = os.path.dirname(os.path.abspath(''))  # project root when run from notebooks-clean/
FIGURES  = os.path.join(ROOT, 'notebooks-clean', 'inference-dim-figures')
PKL_PATH = os.path.join(FIGURES, 'inference-dims-c2st-results.pkl')

with open(PKL_PATH, 'rb') as f:
    all_results = pickle.load(f)

DIMS               = sorted(all_results.keys())          # [4, 8, 16]
DISTANCE_BIN_ORDER = all_results[DIMS[0]]['distance_bin_order']

print('Loaded dims:', DIMS)
print('Bins:', DISTANCE_BIN_ORDER)

In [ ]:
STYLE_MAP = {
    'Uniform':        {'color': '#555555', 'marker': 'o',  'ls': '-',  'lw': 2.5},
    'TailedUniform':  {'color': '#e07b00', 'marker': 's',  'ls': '--', 'lw': 2.5},
    'ExtUniform δ=0.1': {'color': '#2676ae', 'marker': 'D', 'ls': ':',  'lw': 2.0},
    'ExtUniform δ=0.3': {'color': '#0d3b6e', 'marker': 'D', 'ls': ':',  'lw': 2.0},
}

In [ ]:
fig, axes = plt.subplots(1, len(DIMS), figsize=(5 * len(DIMS), 5), sharey=True)

x_pos = np.arange(len(DISTANCE_BIN_ORDER))

for col, dim in enumerate(DIMS):
    ax = axes[col]
    res = all_results[dim]
    model_names = res['model_names']
    offsets = np.linspace(-0.3, 0.3, len(model_names))

    for i, name in enumerate(model_names):
        means, stds, xs = [], [], []
        for j, label in enumerate(DISTANCE_BIN_ORDER):
            vals = np.array(res['c2st_results'][name][label])
            if len(vals):
                means.append(np.mean(vals))
                stds.append(np.std(vals))
                xs.append(x_pos[j] + offsets[i])
        s = STYLE_MAP.get(name, {'color': 'gray', 'marker': 'o'})
        ax.errorbar(xs, means, yerr=stds, fmt=s['marker'],
                    color=s['color'], label=name,
                    capsize=3, capthick=1.5, linewidth=0,
                    elinewidth=1.5, markersize=7, alpha=0.9)

    ax.axvline(4.5, color='red', linestyle='--', linewidth=1.5, alpha=0.8,
               label='Prior boundary')
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Ctr', '0.25', '0.5', '0.75', '1.0', '2σ'],
                       fontsize=9, rotation=30)
    ax.set_title(f'dim={dim}', fontsize=13)
    ax.set_xlabel('Distance from prior center', fontsize=11)
    if col == 0:
        ax.set_ylabel('C2ST vs Reference', fontsize=11)
    if col == len(DIMS) - 1:
        ax.legend(fontsize=8, loc='upper left')

fig.suptitle('Proposal efficiency across dimensions (6000 sims fixed)', fontsize=14)
plt.tight_layout()
os.makedirs(FIGURES, exist_ok=True)
fig.savefig(os.path.join(FIGURES, 'dim-waste-c2st-profiles.pdf'), bbox_inches='tight', dpi=300)
plt.show()
print('Saved dim-waste-c2st-profiles.pdf')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# Collect C2ST at extrapolation bin per (proposal, dim)
EXTRAP_BIN = '2sigma-extrap'

for name, s in STYLE_MAP.items():
    means, stds = [], []
    dims_available = []
    for dim in DIMS:
        if dim not in all_results:
            continue
        vals = np.array(all_results[dim]['c2st_results'].get(name, {}).get(EXTRAP_BIN, []))
        if len(vals):
            means.append(np.mean(vals))
            stds.append(np.std(vals))
            dims_available.append(dim)
    if means:
        ax.errorbar(dims_available, means, yerr=stds,
                    fmt=f"{s['marker']}{s['ls']}",
                    color=s['color'], label=name,
                    capsize=4, capthick=1.5, linewidth=s['lw'],
                    elinewidth=1.5, markersize=8, alpha=0.95)

# Annotate waste fractions for ExtUniform
for delta, delta_key in [(0.1, 'ExtUniform δ=0.1'), (0.3, 'ExtUniform δ=0.3')]:
    for dim in DIMS:
        waste = 1 - (1 / (1 + delta)) ** dim
        if dim not in all_results:
            continue
        vals = np.array(all_results[dim]['c2st_results'].get(delta_key, {}).get(EXTRAP_BIN, []))
        if len(vals):
            ax.annotate(f'{waste*100:.0f}%',
                        xy=(dim, np.mean(vals)),
                        xytext=(3, 4), textcoords='offset points',
                        fontsize=7, color=STYLE_MAP[delta_key]['color'], alpha=0.8)

ax.axhline(0.5, color='gray', linestyle=':', linewidth=2, alpha=0.7, label='Ideal C2ST=0.5')
ax.set_xlabel('Dimensionality', fontsize=13)
ax.set_ylabel('C2ST vs Reference (2σ extrapolation)', fontsize=13)
ax.set_title('Degradation at prior boundary: fixed 6000 sims\n(% = wasted simulations outside prior)', fontsize=12)
ax.set_xticks(DIMS)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES, 'dim-waste-degradation.pdf'), bbox_inches='tight', dpi=300)
plt.show()
print('Saved dim-waste-degradation.pdf')